In [ ]:
import torch
torch.__version__

In [ ]:
!rm -rf kernelforge/
!git clone https://github.com/marcalph/kernelforge.git

In [ ]:
# wurlitzer -> capture c level stdout/err, ninja -> cpp build tool
!pip install wurlitzer ninja torchvision
%load_ext wurlitzer

### python version

In [ ]:
!ls kernelforge/

In [ ]:

import torch, os, math
import torchvision as tv
import torchvision.transforms.functional as tvf
from torchvision import io
import matplotlib.pyplot as plt
from torch.utils.cpp_extension import load_inline



def show_img(x, figsize=(4,3), **kwargs):
    plt.figure(figsize=figsize)
    plt.axis('off')
    if len(x.shape)==3: x = x.permute(1, 2, 0)  # CHW -> HWC
    plt.imshow(x.cpu(), **kwargs)

img = io.read_image('./kernelforge/pisco.jpg') # CHW
img_small = tvf.resize(img, 150, antialias=True)
ch,h,w = img_small.shape

show_img(img_small)


def rgb2grey_py(x):
    c,h,w = x.shape
    n = h*w
    x = x.flatten()
    res = torch.empty(n, dtype=x.dtype, device=x.device)
    for i in range(n): res[i] = 0.2989*x[i] + 0.5870*x[i+n] + 0.1140*x[i+2*n]
    return res.view(h,w)
     




In [ ]:
%%time
img_g = rgb2grey_py(img_small)
     


In [ ]:
img_g = rgb2grey_py(img_small)
print(img_g.shape)
show_img(img_g, cmap='gray')
     


### CUDA

In [ ]:
!rm -rf /root/.cache/torch_extensions/py312_cu128/rgb_to_grayscale_extension/

In [ ]:
!ls /root/.cache/torch_extensions/py312_cu128/rgb_to_grayscale_extension

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING']='1'
from pathlib import Path
import torch
from torchvision.io import read_image, write_png
from torch.utils.cpp_extension import load_inline


def compile_extension():
    cuda_source = Path("kernelforge/src/kernels/grayscale.cu").read_text()
    cpp_source = "torch::Tensor rgb_to_grayscale(torch::Tensor image);"

    return load_inline(
        name="rgb_to_grayscale_extension",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=["rgb_to_grayscale"],
        with_cuda=True,
        extra_cuda_cflags=["-O2"],
        verbose=True,
    )


def main():
    """
    Read input image, convert it to grayscale via custom CUDA kernel and write out as png.
    """

    x = read_image("kernelforge/pisco.jpg").permute(1, 2, 0).cuda()
    print("Input image:", x.shape, x.dtype, "mean:", x.float().mean().item())

    assert x.dtype == torch.uint8

    y = ext.rgb_to_grayscale(x)

    print("Output image:", y.shape, y.dtype, "mean:", y.float().mean().item())
    write_png(y.permute(2, 0, 1).cpu(), "pisco_g.png")

ext = compile_extension()
main()


In [ ]:
imgc = img.contiguous().cuda()
ext

In [ ]:
%%time
res = ext.rgb_to_grayscale(imgc).cpu()
res.shape